# Example code to calculate similarity metrics 

**Disclaimer** The GediSimulator package needs to be downloaded before running any of this code, as well as a folder with all the laz files used and a geojson with the extent of the laz files used.

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkt
from tqdm import tqdm

import sys
sys.path.append('..')

from chap1_modules.gedi_pairs import pair_comparison
from chap1_modules.gedi_simulator_hand import gedi_sim

In [ ]:
from shapely import wkt

def intersect_w_laz_files(file, filename = None):
    if filename != None:
        geo_pairs_raw = gpd.read_file(filename)
    else:
        geo_pairs_raw = file

    geo_pairs_raw['shotnum'] = geo_pairs_raw['shot_num'].astype(np.uint64)

    laz_files = gpd.read_file("{GEOJSON WITH EXTENT OF ALL LAZ FILES USED FOR SIMULATION}") 

    pts_in_laz1 = gpd.sjoin(
        geo_pairs_raw,
        laz_files,
        how="inner",
        predicate="intersects"
    )

    pts_in_laz1['filename'] = ['{DIR WHERE LAZ FILES ARE LOCATED}' + f for f in list(pts_in_laz1.fname)] #/mnt/MDD_HD/
    pts_in_laz2['crs'] = pts_in_laz2.CRS

    geo_pairs = pd.concat([pts_in_laz1, pts_in_laz2])

    geo_pairs['laz_geo'] = [geo_pairs.iloc[[i]].to_crs(geo_pairs.iloc[i].crs).geometry.values[0] for i in range(len(geo_pairs))]

    return geo_pairs

def calculate_pairsim_for_fs(shot_num_2, weight, geo_pairs_als_file):

    row_1 = geo_pairs_als_file[(geo_pairs_als_file['shot_num_2'] == shot_num_2) & (geo_pairs_als_file['weight'] == weight)]

    if len(row_1) == 2:
        try:
            i_1, e_1 = gedi_sim.get_gedisim_wf(
                        row_1.iloc[0].filename,
                        row_1.iloc[0].laz_geo.x,
                        row_1.iloc[0].laz_geo.y,
                        row_1.iloc[0].new_shot_num_1,
                        all_again=True
                    )
            
        except Exception as e:
            print(e, '. Trying again S1 with second point cloud...')
            i_1, e_1 = gedi_sim.get_gedisim_wf(
                        row_1.iloc[1].filename,
                        row_1.iloc[1].laz_geo.x,
                        row_1.iloc[1].laz_geo.y,
                        row_1.iloc[1].new_shot_num_1,
                        all_again=True
                    )
    else:
        i_1, e_1 = gedi_sim.get_gedisim_wf(
                        row_1.iloc[0].filename,
                        row_1.iloc[0].laz_geo.x,
                        row_1.iloc[0].laz_geo.y,
                        row_1.iloc[0].new_shot_num_1,
                        all_again=True
                    )
        
    i_2, e_2 = gedi_sim.get_gedisim_wf(
                        '',
                        0,
                        0,
                        row_1.iloc[0].shot_num_2,
                        all_again=False
                    )
        
    met = pair_comparison.calculate_wf_change_metrics(i_1, i_2, e_1, e_2)

    return met

In [ ]:
big_geo = []

for dist in [40, 100, 200, 400]:

    hp_df = gpd.read_parquet(f'data/fs_stratified_samples/rq_2_fs_pairs_{dist}m.parquet')

    changed = hp_df[hp_df['shot_num_1'] != hp_df['new_shot_num_1']]

    print(len(changed))

    changed_w_laz = intersect_w_laz_files(changed)

    # check for existing output and skip already-processed rows 
    out_path = f'data/fs_stratified_samples/fs_sim_all_{dist}m.parquet'

    if os.path.exists(out_path):
        existing = gpd.read_parquet(out_path)
        done_rows = len(existing)
        print(f"[{dist}m] Resuming: {len(existing)} rows already done, "
                f"{len(changed_w_laz) - len(existing)} remaining.")
    else:
        existing = None
        done_rows = 0
        print(f"[{dist}m] No existing file found, starting fresh.")

    # Pre-populate results from existing file
    wass_dist = list(existing['wass_dist'].values) if existing is not None else []
    mets      = list(existing['metrics'].values)   if existing is not None else []

    # Only iterate over rows not yet processed
    remaining = changed_w_laz.iloc[done_rows:]

    for shot2, w in tqdm(zip(remaining.shotnum.values, remaining.weight.values)):

        try:
            
            met = calculate_pairsim_for_fs(str(shot2), w, gpd.GeoDataFrame(changed_w_laz, geometry="laz_geo"))

            mets.append(met)
            wass_dist.append(met[8])
            
        except Exception as e:
            print(e, f'Error for shot {shot2}. Appending NaNs.')
            mets.append((np.nan,) * 10)
            wass_dist.append(np.nan)

        # Combine already-done rows with newly processed rows so far
        n_done    = done_rows
        n_new     = len(wass_dist) - n_done

        processed = pd.concat([
            existing,
            changed_w_laz.iloc[done_rows:].iloc[:n_new].assign(
                wass_dist=wass_dist[n_done:],
                metrics=mets[n_done:]
            )
        ], ignore_index=True) if existing is not None else \
            changed_w_laz.iloc[:len(wass_dist)].assign(wass_dist=wass_dist, metrics=mets)

        processed[['shot_num', 'strata', 's2_feats', 's1_feats', 'weight',
                   'max_distance', 'wass_dist', 'metrics', 'geometry']] \
            .to_file(out_path, driver='parquet')